RANGE

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


2.9.1+cu130
True
NVIDIA GeForce RTX 5050 Laptop GPU


In [1]:
import sys
sys.path.insert(0, "/D:\\Snowpole Detection\\ultralytics4channel-0a38736761a770f7f7dd80064e20b2d9624eda5b")  # parent of the ultralytics package

import ultralytics
from ultralytics import YOLO

print("Ultralytics module file:", ultralytics.__file__)


Ultralytics module file: d:\Snowpole Detection\cuda-env\Lib\site-packages\ultralytics\__init__.py


In [2]:
import cv2
import shutil
import yaml
import numpy as np
from pathlib import Path

ROOT = Path("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset")

RANGE_ROOT  = ROOT / "range"
LABELS_ROOT = ROOT / "labels"
OUT_ROOT    = ROOT / "hha_only"

(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

def compute_hha4(depth):
    depth = cv2.resize(depth, (1024, 1024))
    depth = depth.astype(np.float32)
    depth[depth == 0] = 1e-3

    # --- H: disparity ---
    disparity = 1.0 / depth
    disparity = cv2.normalize(disparity, None, 0, 255, cv2.NORM_MINMAX)

    # --- H: height (proxy) ---
    height = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)

    # --- A: angle with gravity (approx via gradients) ---
    gx = cv2.Sobel(depth, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(depth, cv2.CV_32F, 0, 1, ksize=3)
    angle = np.arctan2(gy, gx)
    angle = cv2.normalize(angle, None, 0, 255, cv2.NORM_MINMAX)

    # --- 4th channel: raw depth (normalized) ---
    depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)

    hha4 = np.dstack([disparity, height, angle, depth_norm]).astype(np.uint8)
    return hha4

def make_split(split):
    img_out = OUT_ROOT / "images" / split
    lbl_out = OUT_ROOT / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for f in (LABELS_ROOT / split).glob("*.txt"):
        shutil.copy2(f, lbl_out / f.name)

    for p in (RANGE_ROOT / split).glob("*.*"):
        depth = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if depth is None:
            continue
        hha4 = compute_hha4(depth)
        cv2.imwrite(str(img_out / f"{p.stem}.png"), hha4)

for s in ["train", "valid", "test"]:
    make_split(s)

data_yaml = OUT_ROOT / "data.yaml"
cfg = {
    "path": str(OUT_ROOT),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": 1,
    "names": ["snow_pole"],
    "channels": 4
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)


In [1]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolov9t.yaml")

model.train(
    data="4ch_rgbd.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,        # early stopping
    batch=8,
    device=0,
    project="Ablation_4CH11n",
    name="4CH_RGBD11n",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.7 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=4ch_rgbd.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov9t.yaml, momentum=0.937, mosaic=1.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017E13DCD710>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [2]:
import sys, importlib

# Insert local 4-channel ultralytics at beginning of path
local_ul4ch = r"c:\To be shifted\Snowpole Detection\ultralytics4channel"
if local_ul4ch not in sys.path:
    sys.path.insert(0, local_ul4ch)

# Unload any previously loaded ultralytics modules
for m in [k for k in list(sys.modules) if k.split('.')[0] == 'ultralytics']:
    sys.modules.pop(m, None)

# Now import fresh
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics loaded from: {ultralytics.__file__}")
print(f"Version: {ultralytics.__version__}")

print("\nLoading model with yolo11n.yaml...")
model = YOLO("yolo11n.yaml", task="detect")
print(f"Model loaded successfully!")
print(f"Model: {model}")


Ultralytics loaded from: c:\To be shifted\Snowpole Detection\venv\Lib\site-packages\ultralytics\__init__.py
Version: 8.4.6

Loading model with yolo11n.yaml...
Model loaded successfully!
Model: YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
   

HHA Encoding

In [1]:
import os
import cv2
import numpy as np
from scipy.ndimage import sobel

EPS = 1e-6

# -------------------------------
# 1. Normalize range to 0–255
# -------------------------------
def normalize_range(r):
    r = r.astype(np.float32)
    r_min, r_max = np.min(r), np.max(r)
    return ((r - r_min) / (r_max - r_min + EPS) * 255).astype(np.uint8)

# --------------------------------
# 2. Height above ground (approx)
# --------------------------------
def compute_height(range_img):
    # Simple ground estimate: minimum per column
    ground = np.min(range_img, axis=0, keepdims=True)
    height = range_img - ground
    height[height < 0] = 0
    return height

# --------------------------------
# 3. Horizontal disparity
# --------------------------------
def compute_disparity(range_img):
    return 1.0 / (range_img + EPS)

# --------------------------------
# 4. Surface normals & angle
# --------------------------------
def compute_angle(range_img):
    dx = sobel(range_img, axis=1)
    dy = sobel(range_img, axis=0)

    # Approximate normal
    normal_z = np.ones_like(range_img)
    norm = np.sqrt(dx**2 + dy**2 + normal_z**2) + EPS

    nx = dx / norm
    ny = dy / norm
    nz = normal_z / norm

    gravity = np.array([0, 0, 1])
    dot = nz  # since gravity is z-axis
    angle = np.arccos(np.clip(dot, -1, 1))

    return angle

def norm255(x):
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    xmin, xmax = x.min(), x.max()
    if xmax - xmin < EPS:
        return np.zeros_like(x, dtype=np.uint8)
    return ((x - xmin) / (xmax - xmin) * 255).astype(np.uint8)
# --------------------------------
# 5. Convert range → HHA
# --------------------------------
def range_to_hha(range_img):
    h1 = compute_height(range_img)
    h2 = compute_disparity(range_img)
    h3 = compute_angle(range_img)

    def norm255(x):
        return ((x - x.min()) / (x.max() - x.min() + EPS) * 255).astype(np.uint8)

    hha = np.stack([
        norm255(h1),
        norm255(h2),
        norm255(h3)
    ], axis=-1)

    return hha



# --------------------------------
# 6. Process dataset folders
# --------------------------------
def process_split(input_dir, output_dir):
    for root, _, files in os.walk(input_dir):
        for file in files:
            if not file.lower().endswith(('.png', '.jpg', '.jpeg', '.npy', '.tiff', '.exr')):
                continue

            in_path = os.path.join(root, file)
            print("Processing:", in_path)
            # Mirror directory structure
            rel_path = os.path.relpath(root, input_dir)
            out_dir = os.path.join(output_dir, rel_path)
            os.makedirs(out_dir, exist_ok=True)

            # Load range
            if file.endswith('.npy'):
                range_img = np.load(in_path)
            else:
                range_img = cv2.imread(in_path, cv2.IMREAD_UNCHANGED)

            if range_img is None:
                print(f"Skipped unreadable file: {in_path}")
                continue
            if range_img.ndim == 3:
                range_img = range_img[:, :, 0] 
            range_img = range_img.astype(np.float32)

            # Normalize range (requested step)
            # range_norm = normalize_range(range_img)

            # HHA uses physical range, not normalized
            hha = range_to_hha(range_img)

            out_path = os.path.join(
                out_dir,
                os.path.splitext(file)[0] + ".png"
            )

            ok = cv2.imwrite(out_path, hha)
            if not ok:
                print("FAILED TO WRITE:", out_path, hha.dtype, hha.shape)



# --------------------------------
# 7. Run for all splits
# --------------------------------
base = "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\range"
out_base = "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\hha"

for split in ["train", "valid", "test"]:
    process_split(
        os.path.join(base, split),
        os.path.join(out_base, split)
    )

Processing: SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\range\train\image_1.png
Processing: SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\range\train\image_10.png
Processing: SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\range\train\image_100.png
Processing: SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\range\train\image_1000.png
Processing: SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\range\train\image_1001.png
Processing: SnowPole Detection A Comprehensive Dataset for Detecti

In [2]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="3ch_hha.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,        # early stopping
    batch=8,
    device=0,
    project="Ablation_3CH11n",
    name="3CH_HHA11n",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.8 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=3ch_hha.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D69DD54690>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [3]:
import cv2
import numpy as np

img_path = r"SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\range_only\images\train\image_1.png"

img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)

print("Type:", type(img))
print("Dtype:", img.dtype)
print("Shape:", img.shape)
print("Min / Max:", img.min(), img.max())


Type: <class 'numpy.ndarray'>
Dtype: uint8
Shape: (1024, 1024, 4)
Min / Max: 0 255


In [ ]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="4ch_rgbd.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,        # early stopping
    batch=8,
    device=0,
    project="Ablation_4CH11n",
    name="4CH_RGBD11n",
    amp=False,
    augment=False,
    workers=0,
)

In [3]:
print(f"Model task: {model.task}")
print(f"Model type: {type(model.model).__name__}")
print(f"Number of parameters: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"\nReady for training with 4-channel input!")


Model task: detect
Model type: DetectionModel
Number of parameters: 2,624,080

Ready for training with 4-channel input!


In [1]:
import torch
import cv2
import shutil
import yaml
import glob
import numpy as np
from pathlib import Path
import ultralytics
from ultralytics import YOLO

In [ ]:
# ===================== PATHS =====================
ROOT = Path(
    "D:\\Snowpole Detection\\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
    "\\SnowPole_Detection_Dataset"
)

COMB_ROOT   = ROOT / "combined_color"
RANGE_ROOT  = ROOT / "range"
OUT_ROOT   = ROOT / "1ch_range"
LABELS_ROOT = ROOT / "labels"

(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

In [ ]:
def preprocess_range(img):
    img = cv2.resize(img, (1024, 1024))
    img = cv2.equalizeHist(img)
    img = img.astype(np.float32) / 255.0
    return img


In [4]:
RGB_ROOT = ROOT / "3ch_rgb"

(RGB_ROOT / "images").mkdir(parents=True, exist_ok=True)
(RGB_ROOT / "labels").mkdir(parents=True, exist_ok=True)

def make_split(split):
    src_img_dir = COMB_ROOT / split
    dst_img_dir = RGB_ROOT / "images" / split
    dst_lbl_dir = RGB_ROOT / "labels" / split

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels
    for f in (LABELS_ROOT / split).glob("*.txt"):
        shutil.copy2(f, dst_lbl_dir / f.name)

    imgs = list(src_img_dir.glob("*.*"))
    print(f"{split}: {len(imgs)} images")

    for img_path in imgs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        img = cv2.resize(img, (640, 640))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        outpath = dst_img_dir / f"{img_path.stem}.png"
        cv2.imwrite(str(outpath), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


In [5]:
for split in ["train", "valid", "test"]:
    make_split(split)


train: 1367 images
valid: 390 images
test: 197 images


In [6]:
for split in ["train", "valid", "test"]:
    paths = glob.glob(str(RGB_ROOT / f"images/{split}/*.png"))
    for p in paths:
        img = cv2.imread(p)
        if img is None or img.shape[2] != 3:
            print("BAD IMAGE:", p)


In [7]:
data_yaml = RGB_ROOT / "data.yaml"

cfg = {
    "path": str(RGB_ROOT),
    "train": "images/train",
    "val":   "images/valid",
    "test":  "images/test",
    "names": ["snow_pole"],
    "nc": 1
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)


In [8]:
model = YOLO("yolov8n.yaml")

model.train(
    data=str(data_yaml),
    imgsz=640,
    epochs=50,
    batch=2,
    device=0,
    project="SnowPole_3ch",
    name="yolo_rgb",
    amp=False,
    workers=0
)


Ultralytics 8.3.250  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\3ch_rgb\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mos

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000021B537F5450>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# ===============================
# CONFIG
# ===============================
ROOT_DIR = (
    "SnowPole Detection A Comprehensive Dataset for Detection and Localization "
    "Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset"
)
OUTPUT_DIR = "repl_nearir"

MODALITIES = ["nearir", "signal", "range"]
SPLITS = ["train", "valid", "test"]

IMG_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tiff")

# ===============================
# NORMALIZATION RULES
# (no modality normalized)
# ===============================
NORMALIZATION_RULES = {
    "nearir": None,
    "signal": None,
    "range": None
}

# ===============================
# UTILS
# ===============================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def load_single_channel(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise RuntimeError(f"Failed to read image: {path}")
    if img.ndim == 3:
        img = img[..., 0]  # keep first channel only
    return img.astype(np.float32)

def apply_normalization(img, rule):
    # Explicitly do nothing if rule is None
    return img

def to_uint8(img):
    # No semantic normalization, only safe casting
    if img.max() <= 1.0:
        img = img * 255.0
    return np.clip(img, 0, 255).astype(np.uint8)

# ===============================
# CREATE OUTPUT STRUCTURE
# ===============================
for split in SPLITS:
    ensure_dir(os.path.join(OUTPUT_DIR, "images", split))

# ===============================
# DATASET BUILDING (IMAGES ONLY)
# ===============================
for split in SPLITS:
    # use first modality as filename reference
    ref_dir = os.path.join(ROOT_DIR, MODALITIES[0], split)

    image_files = [
        f for f in os.listdir(ref_dir)
        if f.lower().endswith(IMG_EXTENSIONS)
    ]

    for img_name in tqdm(image_files, desc=f"Processing {split}"):

        channels = []

        for modality in MODALITIES:
            img_path = os.path.join(ROOT_DIR, modality, split, img_name)

            img = load_single_channel(img_path)
            img = apply_normalization(img, NORMALIZATION_RULES[modality])

            channels.append(img)

        # Stack → (H, W, 3)
        fused_img = np.stack(channels, axis=-1)
        fused_img = to_uint8(fused_img)

        # Save
        out_path = os.path.join(OUTPUT_DIR, "images", split, img_name)
        cv2.imwrite(out_path, fused_img)

print("\nImage-only repl_reflec dataset created with NO normalization barriers.")


Processing test: 100%|██████████| 197/197 [00:03<00:00, 62.25it/s]


Image-only repl_reflec dataset created with NO normalization barriers.


In [2]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="3ch_replreflec.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,        # early stopping
    batch=8,
    device=0,
    project="3CH_repl_nearir",
    name="3CH_repl_nearir",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.11 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=3ch_replreflec.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mos

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A0AF860DD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [1]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="4ch_rgbd.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,        # early stopping
    batch=8,
    device=0,
    project="3CH_repl_nearir",
    name="3CH_repl_nearir",
    amp=False,
    augment=False,
    workers=0,
)

OSError: [WinError 1455] The paging file is too small for this operation to complete. Error loading "c:\To be shifted\Snowpole Detection\venv\Lib\site-packages\torch\lib\nvperf_host.dll" or one of its dependencies.